# Lab6: YOLO Inference Demo (Colab-Free, curated media)

This notebook runs **object detection** with a pretrained YOLO model.
It automatically downloads **two COCO sample images** (object-rich) and a **short sample video**, then runs inference and displays results. Designed for **free Google Colab**.

In [ ]:
# 0) Optional: Check GPU (not required for a few images)
import subprocess
try:
    subprocess.run(['nvidia-smi'], check=False)
except Exception:
    print('nvidia-smi not available (CPU-only).')


In [ ]:
# 1) Install ultralytics (YOLO)
!pip -q install ultralytics>=8.2.0


In [ ]:
# 2) Load YOLO (small model for speed)
from ultralytics import YOLO
try:
    model = YOLO('yolo11n.pt')
    print('Loaded yolo11n.pt')
except Exception:
    print('yolo11n.pt not found, falling back to yolov8n.pt')
    model = YOLO('yolov8n.pt')
    print('Loaded yolov8n.pt')


## Download exactly 2 object-rich images (COCO samples)

In [ ]:
import os, urllib.request
os.makedirs('test_images', exist_ok=True)

# Stable COCO sample images with many detectable objects
candidate_urls = [
    'http://images.cocodataset.org/val2017/000000039769.jpg',  # zebras (multiple objects)
    'http://images.cocodataset.org/val2017/000000397133.jpg',  # people/bikes
    'http://images.cocodataset.org/val2017/000000522713.jpg',  # traffic scene
    'http://images.cocodataset.org/val2017/000000000285.jpg',  # crowded scene
]

img_paths = []
for i, url in enumerate(candidate_urls):
    if len(img_paths) >= 2:
        break
    try:
        path = f'test_images/img_{i}.jpg'
        urllib.request.urlretrieve(url, path)
        img_paths.append(path)
    except Exception as e:
        print('Failed to download:', url, e)

print('Using images:', img_paths)
assert len(img_paths) == 2, 'Could not fetch two images. Re-run or upload manually.'


## Run inference on the two images

In [ ]:
from PIL import Image
from IPython.display import display

results = model(img_paths, stream=True)
for r in results:
    im_array = r.plot()  # numpy array with boxes drawn
    display(Image.fromarray(im_array))
    if hasattr(r, 'boxes') and r.boxes is not None:
        try:
            print('Classes:', r.boxes.cls.cpu().numpy())
            print('Confidences:', r.boxes.conf.cpu().numpy())
        except Exception:
            pass


## (Optional) Download a short sample video and run detection

In [ ]:
import urllib.request
video_urls = [
    'https://samplelib.com/lib/preview/mp4/sample-5s.mp4',
    'https://filesamples.com/samples/video/mp4/sample_640x360.mp4',
    'https://sample-videos.com/video321/mp4/240/big_buck_bunny_240p_1mb.mp4',
]
video_path = 'sample_video.mp4'
downloaded = False
for url in video_urls:
    try:
        urllib.request.urlretrieve(url, video_path)
        downloaded = True
        print('Downloaded video from:', url)
        break
    except Exception as e:
        print('Failed from', url, e)
if not downloaded:
    print('Could not download any sample video. You can upload your own .mp4 instead.')


## Run YOLO on the sample video (saves annotated output)

In [ ]:
import os
if os.path.exists('sample_video.mp4'):
    # Save predictions under runs/detect/
    model.predict(source='sample_video.mp4', save=True)
    print('Saved results under runs/detect')
else:
    print('No sample video available. Upload a short .mp4 if you want to test.')


## Play the processed video inline (HTML5 player)

In [ ]:
from glob import glob
from IPython.display import HTML

mp4s = sorted(glob('runs/detect/*/*.mp4'))
if mp4s:
    print('Displaying:', mp4s[-1])
    display(HTML(f"<video width='640' height='360' controls loop autoplay src='{mp4s[-1]}'></video>"))
else:
    print('No processed video found. Did detection run successfully?')


**Notes**
- Images are from the official COCO sample server (object-dense). If a URL fails, re-run the cell.
- The inline player uses HTML5; if it does not auto-play, press play.
- For SAR: swap to maritime images and consider fine-tuning.